# Big Data Analysis and Application Experiment 1

**Experiment Topic:** Data Preparation with pandas  
**School:** Hubei University  
**Class:** Software Engineering 2402  
**Student Name:** Qin Tian  
**Student ID:** 202431123002054
**Instructor:** Li Jie


# 1. Problem Description

This experiment uses employee attrition data from pfm_train.csv and pfm_test.csv. The task in this part is not the final prediction model yet. What I need to do first is prepare the raw data for the later logistic regression work.

In the original files, the training set contains the target column Attrition, while the test set does not. So the main problem is not hard to see: before modeling, I need to check the dataset structure, find useless columns, handle text features that can not go into the model directly, and keep the processed train and test data in the same feature format.


# 2. Basic Approach

First, I read the two csv files with pandas and check their shape, column types, missing values, duplicate rows, and constant columns. This step helps me know what kind of cleaning is really needed, instead of changing the data without checking it first.

After the inspection, I remove Over18, StandardHours, and EmployeeNumber, because they do not provide useful information for prediction. Then I use pd.get_dummies() to encode the categorical columns, split the training data into X_train and y_train, and use reindex() to keep the test set aligned with the training feature columns. In the last step, I standardize the original numeric features by StandardScaler, fitting on the training set first and applying the same rule to the test set.


# 3. Source Code

The following code records the full data preparation process used in this experiment.


The code part is kept step by step, so each check and processing action can be seen clearly.


## 3.1 Read the raw csv files

This step reads the two csv files and takes a quick look at the dataset size and the first few rows.


In [1]:
# use pandas to read the csv files
import pandas as pd

# this one is for scaling the numeric features later
from sklearn.preprocessing import StandardScaler

# read train data and test data
train = pd.read_csv("pfm_train.csv")
test = pd.read_csv("pfm_test.csv")

# just check the size first
print("train shape:", train.shape)
print("test shape:", test.shape)

# have a look of the first few rows
train.head()

train shape: (1100, 31)
test shape: (350, 30)


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,37,0,Travel_Rarely,Research & Development,1,4,Life Sciences,77,1,Male,...,3,80,1,7,2,4,7,5,0,7
1,54,0,Travel_Frequently,Research & Development,1,4,Life Sciences,1245,4,Female,...,1,80,1,33,2,1,5,4,1,4
2,34,1,Travel_Frequently,Research & Development,7,3,Life Sciences,147,1,Male,...,4,80,0,9,3,3,9,7,0,6
3,39,0,Travel_Rarely,Research & Development,1,1,Life Sciences,1026,4,Female,...,3,80,1,21,3,3,21,6,11,8
4,28,1,Travel_Frequently,Research & Development,1,3,Medical,1111,1,Male,...,1,80,2,1,2,3,1,0,0,0


## 3.2 Check basic information and missing values

This step checks column types and missing values, because the later cleaning method depends on them.


In [2]:
# check basic info of train set
print("=== train info ===")
train.info()

# also check the test set
print("\n=== test info ===")
test.info()

# see if there is any missing values in train
print("\n=== missing values in train ===")
print(train.isnull().sum())

# same thing for test
print("\n=== missing values in test ===")
print(test.isnull().sum())

=== train info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1100 non-null   int64 
 1   Attrition                 1100 non-null   int64 
 2   BusinessTravel            1100 non-null   object
 3   Department                1100 non-null   object
 4   DistanceFromHome          1100 non-null   int64 
 5   Education                 1100 non-null   int64 
 6   EducationField            1100 non-null   object
 7   EmployeeNumber            1100 non-null   int64 
 8   EnvironmentSatisfaction   1100 non-null   int64 
 9   Gender                    1100 non-null   object
 10  JobInvolvement            1100 non-null   int64 
 11  JobLevel                  1100 non-null   int64 
 12  JobRole                   1100 non-null   object
 13  JobSatisfaction           1100 non-null   int64 
 14  Marit

## 3.3 Check duplicate rows and constant columns

Duplicate rows may affect the later model, and constant columns do not provide real information.


In [3]:
# check duplicate rows first
print("duplicate rows in train:", train.duplicated().sum())
print("duplicate rows in test:", test.duplicated().sum())

# find the columns which only have one value
const_train = [col for col in train.columns if train[col].nunique(dropna=False) == 1]
const_test = [col for col in test.columns if test[col].nunique(dropna=False) == 1]

# print them out
print("constant cols in train:", const_train)
print("constant cols in test:", const_test)

duplicate rows in train: 0
duplicate rows in test: 0
constant cols in train: ['Over18', 'StandardHours']
constant cols in test: ['Over18', 'StandardHours']


## 3.4 Check the label and categorical columns

Here I check the label distribution and the raw text columns, so I can decide the later encoding method.


In [4]:
# see the target label distribution
print("attrition counts:")
print(train["Attrition"].value_counts())

# find all categorical columns in raw data
cat_cols_raw = train.select_dtypes(include="object").columns.tolist()
print("\nraw categorical cols:", cat_cols_raw)

# check each category values
for col in cat_cols_raw:
    print(f"\nvalue counts of {col}:")
    print(train[col].value_counts())

attrition counts:
Attrition
0    922
1    178
Name: count, dtype: int64

raw categorical cols: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime']

value counts of BusinessTravel:
BusinessTravel
Travel_Rarely        787
Travel_Frequently    205
Non-Travel           108
Name: count, dtype: int64

value counts of Department:
Department
Research & Development    727
Sales                     331
Human Resources            42
Name: count, dtype: int64

value counts of EducationField:
EducationField
Life Sciences       462
Medical             337
Marketing           127
Technical Degree     92
Other                63
Human Resources      19
Name: count, dtype: int64

value counts of Gender:
Gender
Male      653
Female    447
Name: count, dtype: int64

value counts of JobRole:
JobRole
Sales Executive              247
Research Scientist           221
Laboratory Technician        205
Manufacturing Director       101
Healthcare Represen

## 3.5 Drop useless columns

According to the previous check, Over18, StandardHours, and EmployeeNumber are removed in this step.


In [5]:
# copy the data first, so the raw one not be changed
train_clean = train.copy()
test_clean = test.copy()

# these columns are not useful, so remove them
# Over18 and StandardHours are basically same value
# EmployeeNumber is just id
drop_cols = ["Over18", "StandardHours", "EmployeeNumber"]

# drop these columns in both train and test
train_clean = train_clean.drop(columns=drop_cols)
test_clean = test_clean.drop(columns=drop_cols)

# check the shape after dropping
print("train_clean shape:", train_clean.shape)
print("test_clean shape:", test_clean.shape)

train_clean shape: (1100, 28)
test_clean shape: (350, 27)


## 3.6 Encode categorical features

Text columns are converted into dummy variables here, so they can be used by the later model.


In [6]:
# get the categorical columns after dropping useless ones
cat_cols = train_clean.select_dtypes(include="object").columns.tolist()
print("categorical cols after drop:", cat_cols)

# do one-hot encoding for categorical features
# drop_first=True to avoid some repeated dummy columns
train_encoded = pd.get_dummies(train_clean, columns=cat_cols, drop_first=True)
test_encoded = pd.get_dummies(test_clean, columns=cat_cols, drop_first=True)

# split x and y from training data
X_train = train_encoded.drop(columns="Attrition")
y_train = train_encoded["Attrition"]

# make test columns same with train columns
# if some column is missing in test, fill it by 0
X_test = test_encoded.reindex(columns=X_train.columns, fill_value=0)

# find bool type columns
bool_cols_train = X_train.select_dtypes(include="bool").columns
bool_cols_test = X_test.select_dtypes(include="bool").columns

# change bool to int, maybe better for model later
X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
X_test[bool_cols_test] = X_test[bool_cols_test].astype(int)

# check shape after encoding
print("encoded train shape:", train_encoded.shape)
print("encoded test shape:", test_encoded.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

categorical cols after drop: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
encoded train shape: (1100, 42)
encoded test shape: (350, 41)
X_train shape: (1100, 41)
X_test shape: (350, 41)


## 3.7 Scale numeric features

Only the original numeric columns are standardized in this step. The dummy columns are left as 0 and 1.


In [7]:
# pick numeric columns for scaling
# do not include the target column
num_cols = train_clean.select_dtypes(exclude="object").columns.tolist()
num_cols.remove("Attrition")

# print these numeric columns
print("scaled numeric cols:", num_cols)

# create the scaler
scaler = StandardScaler()

# fit on train data first, then transform it
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# use the same scaler on test data
X_test[num_cols] = scaler.transform(X_test[num_cols])

# look the processed train data
X_train.head()

scaled numeric cols: ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


,Age,DistanceFromHome,Education,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,0.000101,-1.028598,1.054313,-1.572091,-1.035216,-0.049260,0.240954,-0.104096,-0.671072,0.762229,...,0,0,1,0,0,0,0,0,0,0
1,1.882064,-1.028598,1.054313,1.161260,0.381124,0.853837,0.240954,0.852589,1.720437,0.486513,...,0,0,1,0,0,0,0,0,0,0
2,-0.332010,-0.296263,0.075626,-1.572091,-2.451557,-0.049260,0.240954,-0.086910,-0.671072,2.416525,...,1,0,0,0,0,0,0,0,1,1
3,0.221508,-1.028598,-1.881748,1.161260,-1.035216,1.756934,1.142483,1.327855,-0.671072,0.210797,...,0,0,1,0,0,0,0,1,0,0
4,-0.996233,-1.028598,0.075626,-1.572091,-1.035216,-0.952357,-0.660575,-0.824846,-0.671072,-0.064919,...,1,0,0,0,0,0,0,0,0,0


## 3.8 Final check of prepared data

This step checks the final shape of the processed data and confirms that no missing value is left.


In [8]:
# final check of the shapes
print("final X_train shape:", X_train.shape)
print("final y_train shape:", y_train.shape)
print("final X_test shape:", X_test.shape)

# see if there is still missing values
print("missing values in X_train:", X_train.isnull().sum().sum())
print("missing values in X_test:", X_test.isnull().sum().sum())

# check the final data types
print("\nX_train dtype summary:")
print(X_train.dtypes.value_counts())

final X_train shape: (1100, 41)
final y_train shape: (1100,)
final X_test shape: (350, 41)
missing values in X_train: 0
missing values in X_test: 0

X_train dtype summary:
int64      21
float64    20
Name: count, dtype: int64


## 3.9 Export preprocessing output for Experiment 2

This step saves the processed training features, labels, and test features, so Experiment 2 can continue with model training directly.


In [9]:
from pathlib import Path

# save the preprocessing output for the next experiment
base_dir = Path.cwd().parent
exp2_dir = next(
    p for p in base_dir.iterdir()
    if p.is_dir()
    and p != Path.cwd()
    and (p / "pfm_train.csv").exists()
    and (p / "pfm_test.csv").exists()
    and any(child.suffix.lower() == ".pptx" for child in p.iterdir())
)
exp2_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(exp2_dir / "X_train_preprocessed.csv", index=False)
y_train.to_frame(name="Attrition").to_csv(exp2_dir / "y_train_preprocessed.csv", index=False)
X_test.to_csv(exp2_dir / "X_test_preprocessed.csv", index=False)

print("saved:", exp2_dir / "X_train_preprocessed.csv")
print("saved:", exp2_dir / "y_train_preprocessed.csv")
print("saved:", exp2_dir / "X_test_preprocessed.csv")

saved: d:\daima\cursor\大数据分析\实验二\X_train_preprocessed.csv
saved: d:\daima\cursor\大数据分析\实验二\y_train_preprocessed.csv
saved: d:\daima\cursor\大数据分析\实验二\X_test_preprocessed.csv


# 4. Experimental Data and Results

**Experimental Data:**  
Add screenshots of the original dataset, missing value check, duplicate check, and other key intermediate outputs here.  

**Experimental Results:**  
Add the final running results, shapes of X_train, y_train, X_test, and related screenshots here.


# 5. Reflections on the Experiment

This experiment looked simple at first, but after doing it step by step, I found that the important problems were hidden in small details. At first I thought the main work would be handling missing values. After checking info() and isnull().sum(), I found that both datasets were already complete. So the real problem was not null values, but some columns that looked normal and still should not be kept, such as Over18, StandardHours, and EmployeeNumber.

Another thing I understood more clearly is that get_dummies() is not only changing text into 0 and 1. After encoding, the structure of the table changes too. Because of that, the test set must follow the same feature columns as the training set. The step reindex(columns=X_train.columns, fill_value=0) looked small at first, but it is actually very important. If this step is missed, the later model may still run in some cases, but the feature meaning is already not the same anymore.

This experiment also helped me understand the difference between fit_transform() and transform() in a more practical way. The scaling rule should come from the training set first, and then the same rule should be used on the test set. If the test set also joins the fitting step, the whole preparation process is no longer strict. So after this experiment, I feel data preparation is not just a simple step before modeling. If this part is careless, the later result may look normal, but the process itself is already wrong.

One more thing became clearer after I linked this notebook to Experiment 2. The work in this experiment is not isolated. The processed X_train, y_train, and X_test are exactly the input needed by the later logistic regression task. That means the quality of preprocessing will directly affect model training, evaluation, and prediction in the next experiment. So this experiment is not only about cleaning data. It is also the foundation for the next step of the whole analysis process.
